In [1]:
import numpy as np

def hmm_parameters(pi, A, mus, Sigmas, tol=1e-8):
    pi = np.asarray(pi, dtype=float)
    A = np.asarray(A, dtype=float)
    mus = np.asarray(mus, dtype=float)
    Sigmas = np.asarray(Sigmas, dtype=float)
    if pi.ndim != 1:
        raise ValueError("pi must be a 1D array of shape (K,)")
    K = pi.shape[0]
    if A.shape != (K, K):
        raise ValueError("A must have shape (K, K)")
    if mus.ndim != 2:
        raise ValueError("mus must be a 2D array of shape (K, d)")
    if mus.shape[0] != K:
        raise ValueError("mus must have K rows")
    d = mus.shape[1]
    if Sigmas.shape != (K, d, d):
        raise ValueError("Sigmas must have shape (K, d, d)")
    if np.any(pi < 0):
        raise ValueError("pi must be nonnegative")
    if not np.isclose(pi.sum(), 1.0, atol=tol):
        raise ValueError("pi must sum to 1")
    if np.any(A < 0):
        raise ValueError("A must be nonnegative")
    if not np.allclose(A.sum(axis=1), 1.0, atol=tol):
        raise ValueError("Each row of A must sum to 1")
    eps = 1e-8
    for k in range(K):
        S = Sigmas[k]
        if not np.allclose(S, S.T, atol=tol):
            raise ValueError(f"Sigma[{k}] must be symmetric")
        try:
            np.linalg.cholesky(S + eps * np.eye(d))
        except np.linalg.LinAlgError:
            raise ValueError(f"Sigma[{k}] must be positive definite")
    if not np.all(np.isfinite(mus)):
        raise ValueError("mus contains NaN/inf")
    if not np.all(np.isfinite(Sigmas)):
        raise ValueError("Sigmas contains NaN/inf")
    return {"pi": pi, "A": A, "mus": mus, "Sigmas": Sigmas, "K": K, "d": d}

def prepare_gaussian_cache(mus, Sigmas, jitter=1e-9):
    mus = np.asarray(mus, dtype=float)
    Sigmas = np.asarray(Sigmas, dtype=float)
    K, d = mus.shape
    Ls = np.empty((K, d, d), dtype=float)
    logdets = np.empty(K, dtype=float)
    for k in range(K):
        S = Sigmas[k] + jitter * np.eye(d)
        L = np.linalg.cholesky(S)
        Ls[k] = L
        logdets[k] = 2.0 * np.sum(np.log(np.diag(L)))
    return Ls, logdets

def compute_logB(X, mus, Sigmas, jitter=1e-9):
    X = np.asarray(X, dtype=float)
    mus = np.asarray(mus, dtype=float)
    Sigmas = np.asarray(Sigmas, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be shape (T, d)")
    T, d = X.shape
    K, d2 = mus.shape
    if d != d2:
        raise ValueError("X and mus must have the same feature dimension")
    if Sigmas.shape != (K, d, d):
        raise ValueError("Sigmas must have shape (K, d, d)")
    Ls, logdets = prepare_gaussian_cache(mus, Sigmas, jitter=jitter)
    logB = np.empty((T, K), dtype=float)
    const = d * np.log(2.0 * np.pi)
    for k in range(K):
        XC = X - mus[k]
        Y = np.linalg.solve(Ls[k], XC.T).T
        quad = np.sum(Y * Y, axis=1)
        logB[:, k] = -0.5 * (const + logdets[k] + quad)
    return logB


def forward_filter_from_logB(logB, pi, A):
    logB = np.asarray(logB, dtype=float)
    pi = np.asarray(pi, dtype=float)
    A = np.asarray(A, dtype=float)
    T, K = logB.shape
    phi = np.zeros((T, K), dtype=float)
    c = np.zeros(T, dtype=float)
    logc = np.zeros(T, dtype=float)
    m0 = np.max(logB[0])
    g0 = np.exp(logB[0] - m0)
    alpha0 = pi * g0
    c0 = alpha0.sum()
    if c0 <= 0 or not np.isfinite(c0):
        raise ValueError("Forward filter failed at t=0")
    phi[0] = alpha0 / c0
    c[0] = c0
    logc[0] = m0 + np.log(c0)
    for t in range(1, T):
        phi_pred = phi[t - 1] @ A
        mt = np.max(logB[t])
        gt = np.exp(logB[t] - mt)
        alpha = phi_pred * gt
        ct = alpha.sum()
        if ct <= 0 or not np.isfinite(ct):
            raise ValueError(f"Forward filter failed at t={t}")
        phi[t] = alpha / ct
        c[t] = ct
        logc[t] = mt + np.log(ct)
    loglik = float(np.sum(logc))
    return phi, c, logc, loglik

def backward_normalized_from_logB(logB, A, c):
    logB = np.asarray(logB, dtype=float)
    A = np.asarray(A, dtype=float)
    c = np.asarray(c, dtype=float)
    T, K = logB.shape
    beta_bar = np.ones((T, K), dtype=float)
    for t in range(T - 2, -1, -1):
        m = np.max(logB[t + 1])
        g = np.exp(logB[t + 1] - m)
        tmp = g * beta_bar[t + 1]
        beta_bar[t] = (A @ tmp) / c[t + 1]
        if not np.all(np.isfinite(beta_bar[t])):
            raise ValueError(f"beta_bar contains NaN/inf at t={t}")
    return beta_bar

def smooth_gamma_xi_fast(X, pi, A, mus, Sigmas, jitter=1e-9):
    logB = compute_logB(X, mus, Sigmas, jitter=jitter)
    phi, c, logc, loglik = forward_filter_from_logB(logB, pi, A)
    beta_bar = backward_normalized_from_logB(logB, A, c)
    gamma = phi * beta_bar
    gamma = gamma / gamma.sum(axis=1, keepdims=True)
    T, K = logB.shape
    xi = np.zeros((T - 1, K, K), dtype=float)
    for t in range(T - 1):
        m = np.max(logB[t + 1])
        g = np.exp(logB[t + 1] - m)
        numer = (phi[t, :, None] * A) * (g[None, :] * beta_bar[t + 1][None, :])
        numer = numer / c[t + 1]
        Z = numer.sum()
        if Z <= 0 or not np.isfinite(Z):
            raise ValueError(f"xi normalization failed at t={t}")
        xi[t] = numer / Z
    return phi, beta_bar, gamma, xi, loglik

def m_step_updates(X, gamma, xi, cov_type="full", eps=1e-6):
    X = np.asarray(X, dtype=float)
    gamma = np.asarray(gamma, dtype=float)
    xi = np.asarray(xi, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be shape (T, d)")
    if gamma.ndim != 2:
        raise ValueError("gamma must be shape (T, K)")
    if xi.ndim != 3:
        raise ValueError("xi must be shape (T-1, K, K)")
    T, d = X.shape
    if gamma.shape[0] != T:
        raise ValueError("gamma first dimension must match T")
    K = gamma.shape[1]
    if xi.shape != (T - 1, K, K):
        raise ValueError("xi must be shape (T-1, K, K)")
    pi_new = gamma[0].copy()
    pi_new = pi_new / pi_new.sum()
    numer = xi.sum(axis=0)
    denom = gamma[:-1].sum(axis=0)
    if np.any(denom <= 0) or not np.all(np.isfinite(denom)):
        raise ValueError("A update failed")
    A_new = numer / denom[:, None]
    A_new = np.maximum(A_new, 0.0)
    A_new = A_new / A_new.sum(axis=1, keepdims=True)
    Nk = gamma.sum(axis=0)
    if np.any(Nk <= 0) or not np.all(np.isfinite(Nk)):
        raise ValueError("mu update failed")
    mus_new = (gamma.T @ X) / Nk[:, None]
    Sigmas_new = np.zeros((K, d, d), dtype=float)
    if cov_type == "full":
        for k in range(K):
            xc = X - mus_new[k]
            w = gamma[:, k][:, None]
            S = (w * xc).T @ xc
            S = S / Nk[k]
            S = S + eps * np.eye(d)
            Sigmas_new[k] = 0.5 * (S + S.T)
    elif cov_type == "diag":
        for k in range(K):
            xc = X - mus_new[k]
            w = gamma[:, k][:, None]
            v = (w * (xc ** 2)).sum(axis=0) / Nk[k]
            v = np.maximum(v, eps)
            Sigmas_new[k] = np.diag(v)
    else:
        raise ValueError("cov_type must be 'full' or 'diag'")
    return pi_new, A_new, mus_new, Sigmas_new

def baum_welch_em_fast(X, pi, A, mus, Sigmas, cov_type="full", eps=1e-6, tol=1e-4, max_iter=200, jitter=1e-9):
    X = np.asarray(X, dtype=float)
    pi = np.asarray(pi, dtype=float)
    A = np.asarray(A, dtype=float)
    mus = np.asarray(mus, dtype=float)
    Sigmas = np.asarray(Sigmas, dtype=float)
    _ = hmm_parameters(pi, A, mus, Sigmas)
    loglik_hist = []
    for it in range(max_iter):
        phi, beta_bar, gamma, xi, loglik = smooth_gamma_xi_fast(X, pi, A, mus, Sigmas, jitter=jitter)
        loglik_hist.append(loglik)
        pi_new, A_new, mus_new, Sigmas_new = m_step_updates(X, gamma, xi, cov_type=cov_type, eps=eps)
        if it >= 1:
            improvement = loglik_hist[-1] - loglik_hist[-2]
            if improvement < tol:
                pi, A, mus, Sigmas = pi_new, A_new, mus_new, Sigmas_new
                break
        pi, A, mus, Sigmas = pi_new, A_new, mus_new, Sigmas_new
    _ = hmm_parameters(pi, A, mus, Sigmas)
    return {
        "pi": pi,
        "A": A,
        "mus": mus,
        "Sigmas": Sigmas,
        "loglik_hist": np.asarray(loglik_hist, dtype=float),
    }

def num_params(K, d, cov_type="full"):
    K = int(K)
    d = int(d)
    if K <= 0 or d <= 0:
        raise ValueError("K and d must be positive")
    if cov_type not in ("full", "diag"):
        raise ValueError("cov_type must be 'full' or 'diag'")
    p_pi = K - 1
    p_A = K * (K - 1)
    p_mu = K * d
    if cov_type == "full":
        p_Sigma = K * (d * (d + 1) // 2)
    else:
        p_Sigma = K * d
    return int(p_pi + p_A + p_mu + p_Sigma)


#AIC
def aic(loglik, p):
    loglik = float(loglik)
    p = int(p)
    if not np.isfinite(loglik):
        raise ValueError("loglik must be finite")
    if p < 0:
        raise ValueError("p must be nonnegative")
    return float(-2.0 * loglik + 2.0 * p)

#BIC
def bic(loglik, p, T):
    loglik = float(loglik)
    p = int(p)
    T = int(T)
    if not np.isfinite(loglik):
        raise ValueError("loglik must be finite")
    if p < 0:
        raise ValueError("p must be nonnegative")
    if T <= 1:
        raise ValueError("T must be >= 2")
    return float(-2.0 * loglik + p * np.log(float(T)))

def _random_stochastic_matrix(K, rng, minval=1e-6):
    M = rng.random((K, K)) + minval
    M = M / M.sum(axis=1, keepdims=True)
    return M

def _init_gaussian_params_from_data(X, K, cov_type="full", eps=1e-6, rng=None):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be shape (T, d)")
    T, d = X.shape
    if T < 2:
        raise ValueError("Need T >= 2")
    if K <= 0:
        raise ValueError("K must be positive")
    if cov_type not in ("full", "diag"):
        raise ValueError("cov_type must be 'full' or 'diag'")
    if rng is None:
        rng = np.random.default_rng()
    pi0 = np.ones(K, dtype=float) / float(K)
    A0 = _random_stochastic_matrix(K, rng)
    idx = rng.integers(low=0, high=T, size=K)
    mus0 = X[idx].copy()
    Xc = X - X.mean(axis=0, keepdims=True)
    if cov_type == "full":
        S = (Xc.T @ Xc) / max(T - 1, 1)
        S = S + eps * np.eye(d)
        Sigmas0 = np.repeat(S[None, :, :], K, axis=0)
    else:
        v = (Xc ** 2).mean(axis=0)
        v = np.maximum(v, eps)
        Sigmas0 = np.repeat(np.diag(v)[None, :, :], K, axis=0)
    return pi0, A0, mus0, Sigmas0

def fit_best_of_restarts_fast(
    X,
    K,
    cov_type="full",
    R=10,
    eps=1e-6,
    tol=1e-4,
    max_iter=200,
    jitter=1e-9,
    random_state=0,
    verbose=True,
):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be shape (T, d)")
    T, d = X.shape
    if T < 2:
        raise ValueError("Need T >= 2")
    if R <= 0:
        raise ValueError("R must be positive")

    rng = np.random.default_rng(random_state)
    best_model = None
    best_loglik = -np.inf
    best_r = None
    n_fail = 0
    fail_log = []

    for r in range(R):
        if verbose:
            print(f"Fitting K={K}, restart={r+1}/{R} ...")

        rng_r = np.random.default_rng(rng.integers(0, 2**32 - 1))

        pi0, A0, mus0, Sigmas0 = _init_gaussian_params_from_data(
            X,
            K,
            cov_type=cov_type,
            eps=eps,
            rng=rng_r,
        )
        try:
            model = baum_welch_em_fast(
                X,
                pi0,
                A0,
                mus0,
                Sigmas0,
                cov_type=cov_type,
                eps=eps,
                tol=tol,
                max_iter=max_iter,
                jitter=jitter,
            )
            final_loglik = float(model["loglik_hist"][-1]) if len(model["loglik_hist"]) > 0 else -np.inf

        except Exception as e:
            n_fail += 1
            msg = f"restart={r} {type(e).__name__}: {str(e)[:120]}"
            fail_log.append(msg)
            if verbose:
                print("[FAIL]", msg)
            continue

        if np.isfinite(final_loglik) and final_loglik > best_loglik:
            best_loglik = final_loglik
            best_model = model
            best_r = r

    if verbose:
        print(f"K={K}: failures {n_fail}/{R}")
    if best_model is None:
        preview = "\n".join(fail_log[:5])
        raise RuntimeError(
            "All EM restarts failed. "
            f"First failures:\n{preview}\n"
            "Try increasing eps/jitter, using diag cov, reducing K, or better initialization."
        )
    best_model = dict(best_model)
    best_model["best_loglik"] = float(best_loglik)
    best_model["restart_index"] = int(best_r)
    best_model["n_fail"] = int(n_fail)
    best_model["R"] = int(R)
    best_model["fail_log"] = fail_log
    return best_model

def score_over_K_fast(
    X,
    K_grid,
    cov_type="full",
    R=10,
    eps=1e-6,
    tol=1e-4,
    max_iter=200,
    jitter=1e-9,
    random_state=0,
    verbose=True,
):
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X must be shape (T, d)")
    T, d = X.shape
    if T < 2:
        raise ValueError("Need T >= 2")
    K_grid = list(K_grid)
    if len(K_grid) == 0:
        raise ValueError("K_grid must be non-empty")
    results = []
    models = {}
    for K in K_grid:
        K = int(K)
        if K <= 0:
            raise ValueError("All K must be positive")
        best_model = fit_best_of_restarts_fast(
            X,
            K,
            cov_type=cov_type,
            R=R,
            eps=eps,
            tol=tol,
            max_iter=max_iter,
            jitter=jitter,
            random_state=(random_state + 1000 * K),
            verbose=verbose,
        )
        loglik = float(best_model["best_loglik"])
        p = num_params(K, d, cov_type=cov_type)
        row = {
            "K": K,
            "T": int(T),
            "d": int(d),
            "cov_type": cov_type,
            "p": int(p),
            "loglik": float(loglik),
            "AIC": aic(loglik, p),
            "BIC": bic(loglik, p, T),
            "best_restart": int(best_model["restart_index"]),
            "n_iters": int(len(best_model["loglik_hist"])),
            "failures": int(best_model["n_fail"]),
            "R": int(best_model["R"]),
        }
        results.append(row)
        models[K] = best_model
    return results, models

In [2]:
try:
    from scipy.optimize import linear_sum_assignment  #Hungarian algorithm
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False


def _hungarian_min_cost(cost):
    cost = np.asarray(cost, dtype=float)
    K = cost.shape[0]
    if cost.shape != (K, K):
        raise ValueError("cost must be square (K,K)")

    if _HAVE_SCIPY:
        row_ind, col_ind = linear_sum_assignment(cost)
        perm = np.empty(K, dtype=int)
        perm[row_ind] = col_ind
        return perm

    if K > 8:
        raise RuntimeError("SciPy not available and K>8: Hungarian fallback too slow. Install scipy.")

    import itertools
    best_perm = None
    best_val = np.inf
    for p in itertools.permutations(range(K)):
        v = 0.0
        for i in range(K):
            v += cost[i, p[i]]
        if v < best_val:
            best_val = v
            best_perm = np.array(p, dtype=int)
    return best_perm

#State Alignment

def cost_matrix_mu_l2(mus_a, mus_b):
    mus_a = np.asarray(mus_a, dtype=float)
    mus_b = np.asarray(mus_b, dtype=float)
    if mus_a.ndim != 2 or mus_b.ndim != 2:
        raise ValueError("mus_a and mus_b must be 2D arrays (K,d)")
    if mus_a.shape != mus_b.shape:
        raise ValueError("mus_a and mus_b must have the same shape (K,d)")
    K, d = mus_a.shape
    diff = mus_a[:, None, :] - mus_b[None, :, :]
    cost = np.sqrt(np.sum(diff * diff, axis=2))
    return cost


def _is_diag_matrix(S, atol=1e-12):
    S = np.asarray(S, dtype=float)
    if S.ndim != 2 or S.shape[0] != S.shape[1]:
        return False
    off = S - np.diag(np.diag(S))
    return np.max(np.abs(off)) <= atol


def _sigma_to_diag(Sigmas, atol=1e-12):
    Sigmas = np.asarray(Sigmas, dtype=float)
    if Sigmas.ndim != 3:
        raise ValueError("Sigmas must be (K,d,d)")
    K, d1, d2 = Sigmas.shape
    if d1 != d2:
        raise ValueError("Sigmas must be (K,d,d)")
    diags = np.zeros((K, d1), dtype=float)
    for k in range(K):
        if not _is_diag_matrix(Sigmas[k], atol=atol):
            raise ValueError("Sigmas not diagonal; cannot convert to diag variances safely.")
        diags[k] = np.diag(Sigmas[k])
    return diags


def kl_gaussian_full(mu0, S0, mu1, S1, jitter=1e-9):
    mu0 = np.asarray(mu0, dtype=float)
    mu1 = np.asarray(mu1, dtype=float)
    S0 = np.asarray(S0, dtype=float)
    S1 = np.asarray(S1, dtype=float)
    d = mu0.shape[0]
    if mu1.shape != (d,):
        raise ValueError("mu shapes mismatch")
    if S0.shape != (d, d) or S1.shape != (d, d):
        raise ValueError("Sigma shapes must be (d,d)")
    S1j = S1 + jitter * np.eye(d)
    L1 = np.linalg.cholesky(S1j)
    Y = np.linalg.solve(L1, S0)
    invS1_S0 = np.linalg.solve(L1.T, Y)
    tr_term = np.trace(invS1_S0)
    dm = (mu1 - mu0)
    y = np.linalg.solve(L1, dm)
    quad_term = float(np.dot(y, y))
    S0j = S0 + jitter * np.eye(d)
    L0 = np.linalg.cholesky(S0j)
    logdet1 = 2.0 * np.sum(np.log(np.diag(L1)))
    logdet0 = 2.0 * np.sum(np.log(np.diag(L0)))
    logdet_ratio = float(logdet1 - logdet0)

    kl = 0.5 * (tr_term + quad_term - d + logdet_ratio)
    return float(kl)


def kl_gaussian_diag(mu0, v0, mu1, v1, eps=1e-12):
    mu0 = np.asarray(mu0, dtype=float)
    mu1 = np.asarray(mu1, dtype=float)
    v0 = np.asarray(v0, dtype=float)
    v1 = np.asarray(v1, dtype=float)
    d = mu0.shape[0]
    if mu1.shape != (d,) or v0.shape != (d,) or v1.shape != (d,):
        raise ValueError("diag KL inputs must be (d,)")
    v0 = np.maximum(v0, eps)
    v1 = np.maximum(v1, eps)
    tr_term = float(np.sum(v0 / v1))
    dm = (mu1 - mu0)
    quad_term = float(np.sum((dm * dm) / v1))
    logdet_ratio = float(np.sum(np.log(v1)) - np.sum(np.log(v0)))
    kl = 0.5 * (tr_term + quad_term - d + logdet_ratio)
    return float(kl)


def sym_kl_gaussian(mu0, S0, mu1, S1, cov_type="full", jitter=1e-9):
    """Symmetric KL = 0.5*(KL(p||q)+KL(q||p))"""
    if cov_type == "full":
        k01 = kl_gaussian_full(mu0, S0, mu1, S1, jitter=jitter)
        k10 = kl_gaussian_full(mu1, S1, mu0, S0, jitter=jitter)
        return 0.5 * (k01 + k10)
    elif cov_type == "diag":
        # Convert (d,d) diagonals to (d,) variances if needed
        v0 = np.diag(S0) if S0.ndim == 2 else np.asarray(S0, dtype=float)
        v1 = np.diag(S1) if S1.ndim == 2 else np.asarray(S1, dtype=float)
        k01 = kl_gaussian_diag(mu0, v0, mu1, v1)
        k10 = kl_gaussian_diag(mu1, v1, mu0, v0)
        return 0.5 * (k01 + k10)
    else:
        raise ValueError("cov_type must be 'full' or 'diag'")


def cost_matrix_emission_symkl(model_a, model_b, cov_type="full", jitter=1e-9):
    mus_a = np.asarray(model_a["mus"], dtype=float)
    mus_b = np.asarray(model_b["mus"], dtype=float)
    Sig_a = np.asarray(model_a["Sigmas"], dtype=float)
    Sig_b = np.asarray(model_b["Sigmas"], dtype=float)

    K, d = mus_a.shape
    if mus_b.shape != (K, d):
        raise ValueError("Models must have same (K,d) for mus to align")
    if Sig_a.shape[0] != K or Sig_b.shape[0] != K:
        raise ValueError("Sigmas must have first dimension K")

    cost = np.zeros((K, K), dtype=float)
    for i in range(K):
        for j in range(K):
            cost[i, j] = sym_kl_gaussian(
                mus_a[i], Sig_a[i],
                mus_b[j], Sig_b[j],
                cov_type=cov_type,
                jitter=jitter
            )
    return cost


def apply_permutation_to_model(model, perm):
    perm = np.asarray(perm, dtype=int)
    K = perm.shape[0]

    pi = np.asarray(model["pi"], dtype=float)
    A = np.asarray(model["A"], dtype=float)
    mus = np.asarray(model["mus"], dtype=float)
    Sigmas = np.asarray(model["Sigmas"], dtype=float)

    if pi.shape != (K,):
        raise ValueError("pi shape mismatch with perm")
    if A.shape != (K, K):
        raise ValueError("A shape mismatch with perm")
    if mus.shape[0] != K or Sigmas.shape[0] != K:
        raise ValueError("mus/Sigmas first dim mismatch with perm")

    pi_new = pi[perm]
    mus_new = mus[perm]
    Sigmas_new = Sigmas[perm]
    A_new = A[np.ix_(perm, perm)]

    out = dict(model)
    out["pi"] = pi_new
    out["A"] = A_new
    out["mus"] = mus_new
    out["Sigmas"] = Sigmas_new
    out["perm_applied"] = perm.copy()
    return out


def align_model_to_reference(model, ref_model, method="mu_l2", cov_type="full", jitter=1e-9):
    if method == "mu_l2":
        cost = cost_matrix_mu_l2(ref_model["mus"], model["mus"])
    elif method == "symkl":
        cost = cost_matrix_emission_symkl(ref_model, model, cov_type=cov_type, jitter=jitter)
    else:
        raise ValueError("method must be 'mu_l2' or 'symkl'")
    perm = _hungarian_min_cost(cost)
    aligned = apply_permutation_to_model(model, perm)
    return aligned, perm


#Redundancy Diagnostics

def pairwise_symkl_within_model(model, cov_type="full", jitter=1e-9):
    mus = np.asarray(model["mus"], dtype=float)
    Sig = np.asarray(model["Sigmas"], dtype=float)
    K, d = mus.shape

    M = np.zeros((K, K), dtype=float)
    for i in range(K):
        for j in range(i + 1, K):
            v = sym_kl_gaussian(mus[i], Sig[i], mus[j], Sig[j], cov_type=cov_type, jitter=jitter)
            M[i, j] = v
            M[j, i] = v
    return M


def redundancy_report(model, gamma=None, cov_type="full", jitter=1e-9,
                      kl_thresh=0.5, occ_thresh=None, A_row_thresh=None):
    A = np.asarray(model["A"], dtype=float)
    K = A.shape[0]
    M = pairwise_symkl_within_model(model, cov_type=cov_type, jitter=jitter)
    Nk = None
    T = None
    if gamma is not None:
        gamma = np.asarray(gamma, dtype=float)
        if gamma.ndim != 2 or gamma.shape[1] != K:
            raise ValueError("gamma must be (T,K) matching model K")
        T = gamma.shape[0]
        Nk = gamma.sum(axis=0)

        if occ_thresh is None:
            occ_thresh = 0.01 * float(T)

    flagged = []
    for i in range(K):
        for j in range(i + 1, K):
            if M[i, j] > kl_thresh:
                continue

            if Nk is not None:
                if (Nk[i] > occ_thresh) and (Nk[j] > occ_thresh):

                    continue

            if A_row_thresh is not None:
                dist = float(np.linalg.norm(A[i] - A[j]))
                if dist > A_row_thresh:
                    continue

            flagged.append({
                "i": int(i),
                "j": int(j),
                "symKL": float(M[i, j]),
                "Nk_i": float(Nk[i]) if Nk is not None else None,
                "Nk_j": float(Nk[j]) if Nk is not None else None,
                "A_row_dist": float(np.linalg.norm(A[i] - A[j])) if A_row_thresh is not None else None,
            })

    # sorted by lowest KL first (most redundant)
    flagged.sort(key=lambda x: x["symKL"])

    return {
        "symKL": M,
        "Nk": Nk,
        "occ_thresh": occ_thresh,
        "kl_thresh": kl_thresh,
        "A_row_thresh": A_row_thresh,
        "flagged_pairs": flagged,
    }



#Instability Metrics (between two aligned fits)

def frobenius_A(model1, model2):
    """||A1 - A2||_F"""
    A1 = np.asarray(model1["A"], dtype=float)
    A2 = np.asarray(model2["A"], dtype=float)
    if A1.shape != A2.shape:
        raise ValueError("A shapes must match")
    return float(np.linalg.norm(A1 - A2, ord="fro"))


def mean_state_symkl(model1, model2, cov_type="full", jitter=1e-9):
    mus1 = np.asarray(model1["mus"], dtype=float)
    mus2 = np.asarray(model2["mus"], dtype=float)
    Sig1 = np.asarray(model1["Sigmas"], dtype=float)
    Sig2 = np.asarray(model2["Sigmas"], dtype=float)

    if mus1.shape != mus2.shape:
        raise ValueError("mus shapes must match")
    K = mus1.shape[0]

    vals = np.zeros(K, dtype=float)
    for i in range(K):
        vals[i] = sym_kl_gaussian(mus1[i], Sig1[i], mus2[i], Sig2[i],
                                  cov_type=cov_type, jitter=jitter)
    return float(np.mean(vals)), vals


def map_path_disagreement(gamma1, gamma2):
    g1 = np.asarray(gamma1, dtype=float)
    g2 = np.asarray(gamma2, dtype=float)
    if g1.shape != g2.shape:
        raise ValueError("gamma shapes must match")
    z1 = np.argmax(g1, axis=1)
    z2 = np.argmax(g2, axis=1)
    return float(np.mean(z1 != z2))


def compare_models_instability(model_a, model_b,
                               align_to="a",
                               align_method="mu_l2",
                               cov_type="full",
                               jitter=1e-9,
                               gamma_a=None,
                               gamma_b=None):
    if align_to == "a":
        ref = model_a
        other = model_b
        other_aligned, perm = align_model_to_reference(other, ref, method=align_method,
                                                       cov_type=cov_type, jitter=jitter)
        a_aligned = model_a
        b_aligned = other_aligned
        ga = gamma_a
        gb = gamma_b
        if (gamma_b is not None) and (perm is not None):
            gb = np.asarray(gamma_b, dtype=float)[:, perm]
    elif align_to == "b":
        ref = model_b
        other = model_a
        other_aligned, perm = align_model_to_reference(other, ref, method=align_method,
                                                       cov_type=cov_type, jitter=jitter)
        a_aligned = other_aligned
        b_aligned = model_b
        gb = gamma_b
        ga = gamma_a
        if (gamma_a is not None) and (perm is not None):
            ga = np.asarray(gamma_a, dtype=float)[:, perm]
    else:
        raise ValueError("align_to must be 'a' or 'b'")
    A_frob = frobenius_A(a_aligned, b_aligned)
    mean_kl, per_state_kl = mean_state_symkl(a_aligned, b_aligned, cov_type=cov_type, jitter=jitter)

    out = {
        "perm_used": perm,
        "A_frobenius": float(A_frob),
        "mean_state_symKL": float(mean_kl),
        "per_state_symKL": per_state_kl,
        "aligned_a": a_aligned,
        "aligned_b": b_aligned,
    }
    if (ga is not None) and (gb is not None):
        out["map_disagreement"] = map_path_disagreement(ga, gb)

    return out

def instability_against_reference(models_list, ref_index=0,
                                  align_method="mu_l2", cov_type="full", jitter=1e-9):
    """
    Given a list of fitted models (same K), compute instability vs a reference model.
    Returns list of metric dicts.
    """
    if len(models_list) == 0:
        raise ValueError("models_list must be non-empty")

    ref = models_list[ref_index]
    metrics = []
    for i, m in enumerate(models_list):
        if i == ref_index:
            continue
        comp = compare_models_instability(ref, m, align_to="a",
                                          align_method=align_method,
                                          cov_type=cov_type,
                                          jitter=jitter)
        comp["index"] = int(i)
        metrics.append(comp)
    return metrics